# An Implementation of Modification of safPAKE  #

<h2> This Notebook gives an actual benchmark of our BIO-PAKE </h2>

In [3]:
import numpy as np
import pandas as pd
from bsp import CosineLSH
import json
import python_bulletproofs

In [4]:
import time
import csv
import statistics
import random

# 1 Lib Sodium Initialization and some EC-Operation Functions #

In [5]:
import ctypes
import ctypes.util
import os
import hashlib
import secrets
from typing import List, Tuple

# --- 1. Libsodium Loading & Bindings ---

SCALAR_LEN = 32
POINT_LEN = 32

def load_sodium():
    for name in ("sodium", "libsodium"):
        path = ctypes.util.find_library(name)
        if path:
            try: return ctypes.CDLL(path)
            except: pass
    conda_prefix = os.environ.get("CONDA_PREFIX")
    if conda_prefix:
        candidates = [
            os.path.join(conda_prefix, "Library", "bin", "libsodium.dll"),
            os.path.join(conda_prefix, "lib", "libsodium.so"),
        ]
        for c in candidates:
            if os.path.exists(c): return ctypes.CDLL(c)
    raise OSError("libsodium not found")

sodium = load_sodium()
if hasattr(sodium, "sodium_init"):
    sodium.sodium_init()

# --- Bindings ---

# Scalar Math
sodium.crypto_core_ristretto255_scalar_random.argtypes = [ctypes.c_void_p]
try:
    sodium.crypto_core_ristretto255_scalar_mul.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_scalar_mul.restype = ctypes.c_int
except AttributeError:
    raise RuntimeError("libsodium version too old (missing scalar_mul).")
sodium.crypto_core_ristretto255_scalar_invert.argtypes = [ctypes.c_void_p, ctypes.c_void_p]

# Point Math
sodium.crypto_scalarmult_ristretto255_base.argtypes = [ctypes.c_void_p, ctypes.c_void_p] # P = n * G
sodium.crypto_scalarmult_ristretto255.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p] # P = n * Q

# NEW: Point Addition and Subtraction
try:
    # R = P + Q
    sodium.crypto_core_ristretto255_add.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_add.restype = ctypes.c_int
    
    # R = P - Q
    sodium.crypto_core_ristretto255_sub.argtypes = [ctypes.c_void_p, ctypes.c_void_p, ctypes.c_void_p]
    sodium.crypto_core_ristretto255_sub.restype = ctypes.c_int
except AttributeError:
    raise RuntimeError("libsodium version too old (missing point add/sub).")


# --- Wrappers ---

def random_scalar() -> bytes:
    buf = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_random(ctypes.byref(buf))
    return bytes(buf)

def random_point() -> bytes:
    # To get a random valid point, we generate a random scalar and multiply by base
    return scalar_to_point(random_scalar())

def scalar_mul(x: bytes, y: bytes) -> bytes:
    z = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_mul(ctypes.byref(z), x, y)
    return bytes(z)

def scalar_invert(s: bytes) -> bytes:
    inv = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_invert(ctypes.byref(inv), s)
    return bytes(inv)

def scalar_to_point(s: bytes) -> bytes:
    p = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_scalarmult_ristretto255_base(ctypes.byref(p), s)
    return bytes(p)

def point_mul(scalar: bytes, point: bytes) -> bytes:
    out = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_scalarmult_ristretto255(ctypes.byref(out), scalar, point)
    return bytes(out)

def point_add(p: bytes, q: bytes) -> bytes:
    r = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_core_ristretto255_add(ctypes.byref(r), p, q)
    return bytes(r)

def point_sub(p: bytes, q: bytes) -> bytes:
    r = (ctypes.c_ubyte * POINT_LEN)()
    sodium.crypto_core_ristretto255_sub(ctypes.byref(r), p, q)
    return bytes(r)

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))


def scalar_negate(s: bytes) -> bytes:
    """
    Computes the mathematical negation of a scalar modulo the curve order.
    Returns -s mod q.
    """
    neg = (ctypes.c_ubyte * SCALAR_LEN)()
    sodium.crypto_core_ristretto255_scalar_negate(ctypes.byref(neg), s)
    return bytes(neg)


"""
NOTE: this is super important:
    Since we are adding a paderson commitment now, we need to ensure 

"""

L = (1 << 252) + 27742317777372353535851937790883648493

DALEK_H = python_bulletproofs.get_dalek_default_h()

SCALAR_ONE = b'\x01' + b'\x00' * 31

def int_to_scalar_bytes(val: int) -> bytes:
    """Wraps integers safely around the Curve25519 order and serializes to 32 bytes."""
    L = (1 << 252) + 27742317777372353535851937790883648493
    scalar_int = val % L
    return scalar_int.to_bytes(32, byteorder='little')


# this is important for OPRF's HASH inside!!!!
def hash_point_to_scalar(point_bytes: bytes) -> int:
    h = hashlib.sha256(point_bytes).digest()
    return int.from_bytes(h, 'little') % L

## 1.1 Benchmark on Each EC-operation ##

In [6]:
def run_microbenchmarks(iterations: int = 30) -> pd.DataFrame:
    """
    Measures the average execution time of libsodium EC wrapper functions.
    """
    print(f"=== Starting Microbenchmarks ({iterations} iterations per function) ===")
    
    # 1. Pre-generate valid inputs so we don't accidentally time the setup
    # (Assuming your wrappers are already loaded and working)
    s1 = random_scalar()
    s2 = random_scalar()
    p1 = random_point()
    p2 = random_point()
    
    # Ristretto255 operates on 32-byte chunks, so we create 32 random bytes for XOR
    b1 = os.urandom(32)
    b2 = os.urandom(32)

    # 2. Define the functions to test and their required arguments
    tests = [
        ("random_scalar", random_scalar, ()),
        ("Fixed-Based Multiplication", scalar_to_point, (s1,)),
        ("Variable-Based Multiplication", point_mul, (s1, p1)),
        ("EC Addition", point_add, (p1, p2)),
        ("EC Subtraction", point_sub, (p1, p2))
    ]

    results = []

    # 3. Run the benchmarks
    for func_name, func, args in tests:
        times = []
        for _ in range(iterations):
            start = time.perf_counter()
            func(*args)
            end = time.perf_counter()
            
            times.append(end - start)
            
        # Calculate the average time
        avg_time_sec = sum(times) / iterations
        
        results.append({
            "Function": func_name,
            "Avg Time (Seconds)": avg_time_sec,
            "Avg Time (Microseconds)": avg_time_sec * 1_000_000
        })

    # 4. Format into a pandas DataFrame and sort by slowest to fastest
    df = pd.DataFrame(results)
    df = df.sort_values(by="Avg Time (Microseconds)", ascending=False).reset_index(drop=True)
    
    return df


# Run the 30-iteration benchmark
benchmark_df = run_microbenchmarks(iterations=30)
# Print the beautifully formatted DataFrame
print("\nBenchmark Results (Sorted from slowest to fastest):")
print(benchmark_df.to_string(index=False))

=== Starting Microbenchmarks (30 iterations per function) ===

Benchmark Results (Sorted from slowest to fastest):
                     Function  Avg Time (Seconds)  Avg Time (Microseconds)
Variable-Based Multiplication            0.000146               145.856667
   Fixed-Based Multiplication            0.000091                90.886667
               EC Subtraction            0.000051                50.590000
                  EC Addition            0.000039                39.253333
                random_scalar            0.000002                 1.913333


# 2. Workflow for Client Getting OPRF (with proof) #

## 2.1 Client Side Functionality ##

In [7]:
class FaceAuthenticationClient:
    def __init__(self, facial_vector: np.ndarray, hypervectors: np.ndarray):
        self.facial_vector = facial_vector
        self.hypervectors = hypervectors
        
        self.data_bits = 12
        self.proof_bits = 16
        self.M = (1 << (self.data_bits - 1)) - 1  # 2047 shift for 12-bit
        

#---------------------------------------Saved Computation Results -----------------------------------------------------
        self.face_commitments = []
        self.face_blinding_factors = [] # NEW: We must track the initial blinders!
        self.face_proofs = []

        self.dot_commitments = []
        self.dot_blinding_factors = [] # NEW: Tracking the homomorphic blinders

        self.bit_commitments = []
        self.bit_blinding_factors = []

        self.linkage_commitments = []
        self.linkage_proofs = []

        self.oprf_output= None # For initialization to be None 

#---------------------------------------Saved Computation Results (end) -----------------------------------------------------

        # Pre-compute plaintext LSH bits for the OPRF phase later
        self.raw_dot_products = np.dot(self.hypervectors, self.facial_vector)

        # 0 if Positive, 1 if Negative (Triggers the shift!)
        self.hashed_bits = [0 if dot >= 0 else 1 for dot in self.raw_dot_products]
    

    def CommitFace(self) -> Tuple[List[bytes], List[bytes]]:
        """
        Step 1: Locks the shifted facial vector inside standard Pedersen Commitments (Base G)
        using Python-controlled random blinders so we can track them.
        """
        
        for i, val in enumerate(self.facial_vector):
            shifted_val = int(val) + self.M
            
            if shifted_val < 0 or shifted_val >= (1 << self.data_bits):
                raise ValueError(f"Fatal: Vector value at index {i} is out of bounds.")
            
            # 1. Generate our own random blinder in Python
            r_face = random_scalar()
            self.face_blinding_factors.append(r_face)
            
            # 2. Force the Rust prover to use OUR blinder
            comm, proof = python_bulletproofs.prove_range_with_blinder(
                shifted_val, 
                self.proof_bits, 
                r_face
            )
            
            self.face_commitments.append(comm)
            self.face_proofs.append(proof)
            
        return self.face_commitments, self.face_proofs

    def CreateDotCommitment(self) -> List[bytes]:
        """
        Step 2: Homomorphically computes the dot product of the face commitments 
        and the hypervectors, AND tracks the homomorphic blinders.
        """
        if not self.face_commitments:
            raise ValueError("[Client] Error: Run CommitFace() before CreateDotCommitment()!")
        
        # Curve order L for safely wrapping the blinder math
        L = (1 << 252) + 27742317777372353535851937790883648493

        for row_idx, h_row in enumerate(self.hypervectors):
            dot_product_point = None
            r_dot_int = 0 # Running sum for the blinder
            
            # 1. Compute the homomorphic sum: sum( H_{i,j} * C_j )
            for col_idx in range(len(h_row)):
                h_val = int(h_row[col_idx])
                
                # --- Point Math ---
                C_j = self.face_commitments[col_idx]
                h_scalar = int_to_scalar_bytes(h_val) 
                scaled_C = point_mul(h_scalar, C_j)
                
                if dot_product_point is None:
                    dot_product_point = scaled_C
                else:
                    dot_product_point = point_add(dot_product_point, scaled_C)
                    
                # --- Blinder Math ---
                # We homomorphically calculate r_dot = sum(h_val * r_face) mod L
                r_face_int = int.from_bytes(self.face_blinding_factors[col_idx], 'little')
                r_dot_int = (r_dot_int + h_val * r_face_int) % L
            
            # 2. Compute the offset scalar: M * sum(H_{i,j})
            sum_h = sum(int(h) for h in h_row)
            offset_scalar_bytes = int_to_scalar_bytes(self.M * sum_h)
            
            # 3. Create the offset point using the standard Basepoint G
            offset_point = scalar_to_point(offset_scalar_bytes)
            
            # 4. Subtract the offset to get the clean dot product commitment
            final_dot_commitment = point_sub(dot_product_point, offset_point)
            
            # Save the point AND the calculated blinder!
            self.dot_commitments.append(final_dot_commitment)
            self.dot_blinding_factors.append(r_dot_int.to_bytes(32, byteorder='little'))

        return self.dot_commitments
    
    def CreateBitCommitment(self, GLOBAL_H: bytes) -> List[bytes]:
        """
        Step 3: Creates Pedersen commitments for the extracted LSH bits (0 or 1).
        Formula: C_bit = b_i * G + r_i * H
        """
        
        for b_val in self.hashed_bits:
            # 1. Generate a fresh, mathematically secure random scalar (r_i)
            r_i = random_scalar()
            self.bit_blinding_factors.append(r_i)
            
            # 2. Compute the value point: b_i * G
            # (If b_val is 0, this results in the identity point. If 1, it results in G)
            b_scalar = int_to_scalar_bytes(b_val)
            val_point = scalar_to_point(b_scalar) # Uses standard basepoint G
            
            # 3. Compute the blinding point: r_i * H
            # We use your generated GLOBAL_H to ensure it is perfectly orthogonal to G
            blind_point = point_mul(r_i, GLOBAL_H)
            
            # 4. Add them together: C_bit = (b_i * G) + (r_i * H)
            c_bit = point_add(val_point, blind_point)
            
            self.bit_commitments.append(c_bit)
            
        return self.bit_commitments

    def CreateLinkageProof(self) -> Tuple[List[bytes], List[bytes]]:
        """
        Step 4: Computes C_test = C_d + 2^32 * C_b, and generates a 32-bit 
        Bulletproof to prove the sign bit correctly matches the dot product.
        """
        if not hasattr(self, 'dot_commitments') or not hasattr(self, 'bit_commitments'):
            raise ValueError("[Client] Error: Must compute dot and bit commitments first!")
        
        # Pre-compute the 2^32 shift
        scalar_shift_int = 1 << 32
        scalar_shift_bytes = int_to_scalar_bytes(scalar_shift_int)
        
        # The Curve25519 order (L) for safely wrapping blinders
        L = (1 << 252) + 27742317777372353535851937790883648493

        for i in range(len(self.dot_commitments)):
            C_d = self.dot_commitments[i]
            C_b = self.bit_commitments[i]
            
            # --- 1. Homomorphic Commitment Math ---
            shift_point = point_mul(scalar_shift_bytes, C_b)
            C_test = point_add(C_d, shift_point)
            self.linkage_commitments.append(C_test)
            
            # --- 2. Calculate the True Plaintext Value (v_test) ---
            true_dot = int(self.raw_dot_products[i])
            bit_val = self.hashed_bits[i]
            
            # v_test = d_i + (b_i * 2^32)
            # If honesty is maintained, this stays perfectly within [0, 2^32 - 1]
            v_test = true_dot + (bit_val << 32)
            
            # --- 3. Calculate the Exact Composite Blinder (r_test) ---
            # Extract the raw integer blinders we saved earlier
            r_d_int = int.from_bytes(self.dot_blinding_factors[i], 'little')
            r_b_int = int.from_bytes(self.bit_blinding_factors[i], 'little')
            
            # r_test = (r_dot + 2^32 * r_bit) mod L
            r_test_int = (r_d_int + scalar_shift_int * r_b_int) % L
            r_test_bytes = r_test_int.to_bytes(32, byteorder='little')
            
            # --- 4. Generate the 32-bit Proof! ---
            # We pass our composite blinder into Rust so the proof mathematically binds to C_test
            _, proof = python_bulletproofs.prove_range_with_blinder(
                v_test, 
                32, 
                r_test_bytes
            )
            
            self.linkage_proofs.append(proof)

        return self.linkage_commitments, self.linkage_proofs

    def EvaluateOPRF(self, R_list: List[bytes], S0_list: List[int], S1_list: List[int], G_out: bytes) -> bytes:
        """
        Client Step 5: Evaluates the OT tuples to resolve the OPRF output without 
        revealing which bits were extracted. (Strictly Additive)
        """
        L = (1 << 252) + 27742317777372353535851937790883648493
        M = len(self.hashed_bits)
        
        # FIX: Start a running sum at 0, NOT a product at 1
        obtained_sum = 0
        
        for i in range(M):
            b_i = self.hashed_bits[i]
            r_i_bytes = self.bit_blinding_factors[i]
            R_i = R_list[i]
            
            # 1. Compute the shared secret point
            shared_point = point_mul(r_i_bytes, R_i)
            shared_hash = hash_point_to_scalar(shared_point)
            
            # 2. Extract the payload based on the bit
            if b_i == 0:
                extracted_val = (S0_list[i] - shared_hash) % L
            else: # b_i == 1
                extracted_val = (S1_list[i] - shared_hash) % L
                
            # 3. FIX: ADD into the running sum modulo L
            obtained_sum = (obtained_sum + extracted_val) % L
            
        # --- Final OPRF Output ---
        # Output = G_out + (obtained_sum * G)
        sum_scalar_bytes = int_to_scalar_bytes(obtained_sum)
        sum_point = scalar_to_point(sum_scalar_bytes)
        
        final_oprf_point = point_add(G_out, sum_point)
        
        self.oprf_output = final_oprf_point
        return final_oprf_point

## 2.2 Server Side Functionality ##

In [8]:
class FaceAuthenticationServer:
    def __init__(self, hypervectors: np.ndarray):
        """
        Initializes the server with the public LSH hypervectors.
        Sets up the exact same mathematical bounds as the client.
        """

        self.hypervectors = hypervectors
        
        # Cryptographic configuration (Must match client perfectly)
        self.data_bits = 12
        self.proof_bits = 16
        self.M = (1 << (self.data_bits - 1)) - 1  # 2047 shift for 12-bit
        
# -------------------------------------------- Server State Storage ------------------------------------------------------
        self.client_face_commitments = []
        self.dot_commitments = []
        self.bit_commitments =[]

# -------------------------------------------- Server State Storage (end) ------------------------------------------------------

        # Initialize the OPRF Server's key and shifts
        self.k_0 = int.from_bytes(random_scalar(), 'little')
        num_hyperplanes = len(self.hypervectors)
        self.k_keys = [int.from_bytes(random_scalar(), 'little') for _ in range(num_hyperplanes)]


    def VerifyFaceCommitments(self, commitments: List[bytes], proofs: List[bytes]) -> bool:
        """
        Server Step 1: Receives the face commitments and 16-bit range proofs from the client.
        Verifies that the hidden face vector values strictly fit within the 12-bit shifted bounds.
        """
        if len(commitments) != len(proofs):
            print("[Server] ERROR: Number of commitments does not match number of proofs.")
            return False
            
        print(f"\n[Server] Verifying {len(commitments)} Bulletproofs (16-bit capacity)...")
        
        for i in range(len(commitments)):
            # verify_range automatically uses the Dalek default basepoints (G and H)
            # This perfectly matches what the client used in CommitFace()
            is_valid = python_bulletproofs.verify_range(commitments[i], proofs[i], self.proof_bits)
            
            if not is_valid:
                print(f"[Server] 🚨 REJECTED: Zero-Knowledge Proof at index {i} failed!")
                return False
        
        # Save the valid commitments into the Server's state so we can use them later
        self.client_face_commitments = commitments
        
        return True
    
    def CreateDotCommitment(self) -> List[bytes]:
        """
        Server Step 2: Homomorphically computes the dot product of the verified 
        face commitments and the hypervectors, stripping the offset perfectly.
        """
        if not self.client_face_commitments:
            raise ValueError("[Server] Error: Must run VerifyFaceCommitments successfully first!")

        self.dot_commitments = []

        for row_idx, h_row in enumerate(self.hypervectors):
            dot_product_point = None
            
            # 1. Compute the homomorphic sum: sum( H_{i,j} * C_j )
            for col_idx in range(len(h_row)):
                h_val = int(h_row[col_idx])
                C_j = self.client_face_commitments[col_idx]
                
                h_scalar = int_to_scalar_bytes(h_val) 
                scaled_C = point_mul(h_scalar, C_j)
                
                if dot_product_point is None:
                    dot_product_point = scaled_C
                else:
                    dot_product_point = point_add(dot_product_point, scaled_C)
            
            # 2. Compute the offset scalar: M * sum(H_{i,j})
            sum_h = sum(int(h) for h in h_row)
            offset_scalar_bytes = int_to_scalar_bytes(self.M * sum_h)
            
            # 3. Create the offset point using the standard Basepoint G
            offset_point = scalar_to_point(offset_scalar_bytes)
            
            # 4. Subtract the offset to get the clean dot product commitment
            final_dot_commitment = point_sub(dot_product_point, offset_point)
            
            self.dot_commitments.append(final_dot_commitment)

        return self.dot_commitments
    
    def ReceiveBitCommitments(self, bit_commitments: List[bytes]):
        """
        Server Step 3: Receives the LSH bit commitments (C_b) from the client.
        The server does NOT verify these yet, it just stores them for the linkage step.
        """
        if len(bit_commitments) != len(self.hypervectors):
            raise ValueError("[Server] Error: Number of bit commitments does not match hyperplanes.")
            
        self.bit_commitments = bit_commitments

    def VerifyLinkageProofs(self, linkage_proofs: List[bytes]) -> bool:
        """
        Server Step 4: Homomorphically computes the linkage commitment C_test = C_d + 2^32 * C_b.
        Then, uses the client's provided 32-bit proofs to verify the math holds true.
        """
            
        if len(linkage_proofs) != len(self.dot_commitments):
            print("[Server] 🚨 ERROR: Number of proofs does not match number of commitments.")
            return False
        
        # Pre-compute the 2^32 shift scalar for the curve math
        scalar_shift_bytes = int_to_scalar_bytes(1 << 32)
        
        for i in range(len(self.dot_commitments)):
            C_d = self.dot_commitments[i]
            C_b = self.bit_commitments[i]
            
            # --- 1. Server independently computes 2^32 * C_b ---
            shift_point = point_mul(scalar_shift_bytes, C_b)
            
            # --- 2. Server independently computes C_test = C_d + (2^32 * C_b) ---
            # This is the genius step: The server builds the commitment itself,
            # guaranteeing it is perfectly bound to the true face data.
            C_test = point_add(C_d, shift_point)
            
            # --- 3. Verify the client's 32-bit proof against the SERVER'S computed commitment ---
            # We can use the standard verify_range here because the verifier in Bulletproofs
            # does not need to know the blinder, it only needs the final commitment!
            is_valid = python_bulletproofs.verify_range(C_test, linkage_proofs[i], 32)
            
            if not is_valid:
                print(f"[Server] 🚨 REJECTED: Linkage Proof at index {i} failed! The client lied about their LSH bit.")
                return False
                
        return True
    
    def GenerateOPRFData(self, DALEK_H: bytes) -> tuple:
        M = len(self.bit_commitments)
        R_list, S0_list, S1_list = [], [], []
        
        G_point = scalar_to_point(int_to_scalar_bytes(1))
        
        # FIX: We use an additive sum instead of a product!
        delta_sum = 0

        for i in range(M):
            C_bi = self.bit_commitments[i]
            k_i = self.k_keys[i]
            
            rou_i_bytes = random_scalar()
            delta_i = int.from_bytes(random_scalar(), 'little')
            
            # Keep a running sum of all deltas modulo L
            delta_sum = (delta_sum + delta_i) % L
            
            # --- R_i = rou_i * H ---
            R_i = point_mul(rou_i_bytes, DALEK_H)
            R_list.append(R_i)
            
            # --- S_{i,0} Math ---
            P0 = point_mul(rou_i_bytes, C_bi)
            hash0 = hash_point_to_scalar(P0)
            S_i0 = (hash0 + delta_i) % L
            S0_list.append(S_i0)
            
            # --- S_{i,1} Math ---
            C_bi_minus_G = point_sub(C_bi, G_point)
            P1 = point_mul(rou_i_bytes, C_bi_minus_G)
            hash1 = hash_point_to_scalar(P1)
            
            # FIX: Pure addition here! (delta_i + k_i)
            S_i1 = (hash1 + delta_i + k_i) % L
            S1_list.append(S_i1)
            
        # --- Prepare G_out = (k_0 - SUM(delta_i)) * G ---
        offset_scalar_int = (self.k_0 - delta_sum) % L
        G_out = scalar_to_point(int_to_scalar_bytes(offset_scalar_int))
        
        return R_list, S0_list, S1_list, G_out
    
    def rawPRF(self, hash_int: int):
        L = (1 << 252) + 27742317777372353535851937790883648493
        M = len(self.k_keys)
        
        # Initialize the accumulator with the base key k_0
        accumulator = self.k_0
        
        for i in range(M):
            # Bitwise Magic: Shift the integer right by 'i' positions, 
            # then mask it with '& 1' to isolate exactly that bit (0 or 1).
            shift_amount = (M - 1) - i
            bit_is_set = (hash_int >> shift_amount) & 1       
            
            if bit_is_set == 1:
                # ADDITION ROUTE: We add the key to the running sum
                accumulator = (accumulator + self.k_keys[i]) % L
                
                # NOTE: If you are using the MULTIPLICATIVE route instead, 
                # change the initialization above to accumulator = self.k_0 
                # and use this line instead:
                # accumulator = (accumulator * self.k_keys[i]) % L

        # Map the final accumulated scalar to the curve using basepoint G
        prf_scalar_bytes = int_to_scalar_bytes(accumulator)
        final_prf_point = scalar_to_point(prf_scalar_bytes)
        
        return final_prf_point

## 2.3 BioPAKE Server ##

### 2.3.1 Helper Class: AES Encryption ###

In [9]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.backends import default_backend

In [10]:
class AuthenticatedEncryption:
    """
    Implements the Authenticated Encryption scheme E (Enc, Dec)
    required for safPAKE using AES-256-GCM.
    """

    @staticmethod
    def derive_aes_key(ek: bytes) -> bytes:
        """Derives a 256-bit AES key from the OPRF output's ek using HKDF."""
        # Use a fixed salt and info string for deterministic key derivation
        # The key ek must be K bytes (32 bytes) long.
        salt = b'safpake_key_salt'
        info = b'safpake_aes_key_derivation'
        
        # HKDF is used to expand the PRF key 'ek' into a suitable 256-bit AES key
        return HKDF(
            algorithm=hashes.SHA256(),
            length=16, # AES-256 requires a 32-byte key
            salt=salt,
            info=info,
            backend=default_backend()
        ).derive(ek)

    @staticmethod
    def Encrypt(ek, data: bytes) -> bytes:
        """
        Encrypts data using AES-ECB.
        
        Args:
            data (bytes): The plaintext data to encrypt.
            
        Returns:
            bytes: The encrypted ciphertext.
        """
        key=AuthenticatedEncryption.derive_aes_key(ek)

        # We create a new cipher object for every operation to reset state
        cipher = AES.new(key, AES.MODE_ECB)
        
        # Pad the data to be a multiple of the block size (16 bytes)
        # standard PKCS7 padding is used here.
        padded_data = pad(data, AES.block_size)
        
        return cipher.encrypt(padded_data)

    @staticmethod
    def Decrypt(ek, ciphertext: bytes) -> bytes:
        """
        Decrypts data using AES-ECB.
        
        Args:
            ciphertext (bytes): The encrypted data to decrypt.
            
        Returns:
            bytes: The original plaintext.
        """
        key=AuthenticatedEncryption.derive_aes_key(ek)

        cipher = AES.new(key, AES.MODE_ECB)
        
        # Decrypt and then remove the padding
        padded_plaintext = cipher.decrypt(ciphertext)
        plaintext = unpad(padded_plaintext, AES.block_size)
        
        return plaintext

### 2.3.2 A verification process for a single BAG ###

In [29]:
class PAKE_Bag:
    """
    Manages a single disjoint bag for a fuzzy-PAKE biometric authentication scheme.
    
    Args: 
        num_hyperplanes (int): The number of LSH bits (M).
        tolerance (int): The maximum Hamming distance allowed in the error ball.
    """

    def __init__(self, num_hyperplanes: int, tolerance: int):
        
        # User Database for the error ball. 
        self.DB= dict()

        # AES-256 requires a 32-BYTE key (256 bits).
        self.AES_key_bytes = 32

        # The cosine LSH functionality (generates float hypervectors)
        self.LSH_hash = CosineLSH(num_hyperplanes, 128)

        # The Hamming distance threshold
        self.tolerance = tolerance

        # --- CRITICAL: Quantize Hyperplanes before giving them to the ZK Server ---
        # Assuming your quantization bounds are [-1.0, 1.0] scaled to 11-bits + sign (2047)
        quantized_hyperplanes = CosineLSH.quantize_array(self.LSH_hash.hyperplanes, 12)
        # Initialize the zero-knowledge server with mathematically safe integers
        self.bag_authenticator = FaceAuthenticationServer(quantized_hyperplanes)

        self.AE=AuthenticatedEncryption()

        # initialze the master ciphertext
        # - pack the oprfKey into bytestring
        # - Generate a master key (discard this later)
        # - Encrypt the heavy payload to create the Master Ciphertext 
        server_keys_payload = self._pack_server_keys_to_bytes()
        self.master_key = os.urandom(32)
        self.master_cipher = self.AE.Encrypt(self.master_key, server_keys_payload)

        #Note that since we only need to commit to face once, we can save this from the higher level
        self.face_comms=None

    def _pack_server_keys_to_bytes(self) -> bytes:
        """
        Concatenates the FaceAuthenticationServer's k_0 and k_keys into a single byte string.
        Each scalar is guaranteed to be 32 bytes long (little-endian).
        """
        # 1. Pack k_0 (32 bytes)
        packed_bytes = self.bag_authenticator.k_0.to_bytes(32, byteorder='little')
        
        # 2. Pack each k_i (32 bytes each)
        for k_i in self.bag_authenticator.k_keys:
            packed_bytes += k_i.to_bytes(32, byteorder='little')
            
        return packed_bytes

    def derive_variants(self, orig_face_vec: np.ndarray, base_hash: int) -> list:
        """
        Derives all possible bit-flip combinations for the least reliable hyperplanes.
        """
        # 1. Ask the LSH which hyperplanes are too close to the boundary (using 12-bit quantization)
        indices = self.LSH_hash.filter_by_hyperplane(orig_face_vec, self.tolerance, k_bits=12)
        
        # 2. Reverse the index because CosineLSH hashes MSB-first
        shifts = [int((self.LSH_hash.num_bits - 1) - idx) for idx in indices]
        
        # 3. Prepare the base template: Clear the bits at the unreliable positions
        base_template = base_hash
        for shift in shifts:
            base_template &= ~(1 << shift)  
            
        variants = []
        num_combinations = 1 << len(shifts) 

        # 4. Generate all combinations using a binary counter
        for i in range(num_combinations):
            current_variant = base_template
            for j, shift in enumerate(shifts):
                if (i >> j) & 1:
                    current_variant |= (1 << shift)
                    
            variants.append(current_variant)

        return variants

    def register(self, face_vector: np.ndarray):
        
        # 1. Hash the face using 12-bit quantization
        raw_base_hash = self.LSH_hash.hash(face_vector, k_bits=12)
        
        # FIX: Invert the hash to match the ZKP "Sign-Bit" logic (0 for positive)
        full_mask = (1 << self.LSH_hash.num_bits) - 1
        base_hash = raw_base_hash ^ full_mask

        # --- Serialize the raw facial vector to bytes ---
        face_bytes = face_vector.astype(np.float32).tobytes()

        # print(self.bag_authenticator.rawPRF(base_hash)) for debug use only
        
        # 2. Derive all variants (using the newly aligned base_hash)
        all_variants = self.derive_variants(face_vector, base_hash)

        for pw_prime in all_variants:
            
            # B. Create payload_i = (pristine_base_hash || master_key)
            # This ensures the client recovers the exact original template!
            payload_i = face_bytes + self.master_key
            
            # C. Evaluate the direct PRF using the Server's persistent keys
            # (Returns the 32-byte Elliptic Curve point)
            prf_point_bytes = self.bag_authenticator.rawPRF(pw_prime)
            
            # D. Key Derivation Function (KDF) using Domain Separation
            # Derive the dictionary Tag (as a hex string for easy JSON/dict storage)
            tag_bytes = hashlib.sha256(b"TAG" + prf_point_bytes).digest()
            tag_str = tag_bytes.hex()
            
            # Derive the 32-byte symmetric encryption key (ek_i)
            ek_i = hashlib.sha256(b"KEY" + prf_point_bytes).digest()
            
            # E. Encrypt payload_i using ek_i
            ciphertext = self.AE.Encrypt(ek_i, payload_i)
            
            # F. Store the entry in the Disjoint Bag's Database
            self.DB[tag_str] = ciphertext
        
        return True
    
######----------------------------------------This Begins the Verification Routine ---------------------------------------------------------------
    
    def Check_Input(self, Authentication_Client) -> bool:
        """
        Simulates the network protocol between the Client and the Server.
        Executes the 4-step Zero-Knowledge LSH verification routine.
        
        Returns True if the client proves they hold valid LSH bit commitments 
        derived from valid 12-bit facial vectors.
        """
        print("\n[Protocol] 🔄 Starting Zero-Knowledge Verification...")

        # =====================================================================
        # STEP 1: Commit to the Face Vector
        # =====================================================================
        face_comms, face_proofs = Authentication_Client.CommitFace()
        is_face_valid = self.bag_authenticator.VerifyFaceCommitments(face_comms, face_proofs)
        
        if not is_face_valid:
            print("[Server] ❌ REJECTED: Invalid face commitments or out-of-bounds data.")
            return False

        # =====================================================================
        # STEP 2: Homomorphic Dot Products
        # =====================================================================

        Authentication_Client.CreateDotCommitment()

        self.bag_authenticator.CreateDotCommitment()

        # =====================================================================
        # STEP 3: Bit Commitments
        # =====================================================================

        bit_comms = Authentication_Client.CreateBitCommitment(python_bulletproofs.get_dalek_default_h())
        
        self.bag_authenticator.ReceiveBitCommitments(bit_comms)

        # =====================================================================
        # STEP 4: Linkage Proofs (The 32-bit Shift)
        # =====================================================================
        linkage_comms, linkage_proofs = Authentication_Client.CreateLinkageProof()

        # The server uses the bit commitments and dot commitments it already holds
        # to mathematically rebuild the test commitment and verify the proof!
        is_linkage_valid = self.bag_authenticator.VerifyLinkageProofs(linkage_proofs)

        if not is_linkage_valid:
            print("[Server] ❌ REJECTED: Linkage proofs failed. Bits do not match dot products.")
            return False

        return True

In [30]:
class PAKE_server:
    # this is the high level server function that manage stuff at a high level

    def __init__(self, disjoint_bags:int, number_of_hyperplanes: int, tolerance:int):
        """
        Args: 
            1. Disjoint_bags: number of distinct bags (or) we want to test on
            2. Number_of_hyperplanes: just how many result hashed bits
            3. Tolerance: How many closest hyperplanes I want to filter out 
        """
        # 1. Create a bunch of disjoint bags
        self.bags=[]
        for i in range(disjoint_bags):
            self.bags.append(PAKE_Bag(number_of_hyperplanes,tolerance))
    
    def register(self,face_vec):
        # register the facial vector on each of the bag
        result=True
        for each in self.bags:
            result=result and each.register(face_vec)
        
        return result
    
    def authenticate(self,client):
        server=self
        # takes in a pake client and go authenticate with each of the bag
        result=False
        for each in self.bags:
            result = result or client.verify(each,server)
        
        return result

## 2.4 The Client Class ##

In [31]:
class PAKE_client:
    """Creates a single client for a single PAKE disjoint bag."""
    
    def __init__(self, face_vec: np.ndarray):
        # 1. Quantize the Client's Face Vector to 12 bits

        self.face = CosineLSH.quantize_array(face_vec, 12)

        # These are not yet initialized.
        # The Server will send its public LSH parameters to the Client to initialize.
        self.LSH_hash = None
        self.Face_Authenticator = None # this is the previous authentication client class
        self.sessonOPRF=None
    
    def Initialize(self, LSH):
        """
        Initializes the client using the server's public LSH parameters.
        Applies strict 12-bit quantization to match the Server's ZK-math limits.
        """
        self.LSH_hash = LSH
        
        # 2. Quantize the Server's Hyperplanes
        quantized_hyperplanes = CosineLSH.quantize_array(LSH.hyperplanes, 12)
        
        # 3. Safely initialize the ZKP backend
        self.Face_Authenticator = FaceAuthenticationClient(
            self.face, 
            quantized_hyperplanes
        )

        # 4. get the OPRF output by first proving the thing 
    def getOPRF(self, Authentication_Bag):
        """
        Authenticates a client by first verifying their Zero-Knowledge Proofs,
        then executing the Oblivious Transfer (OT) to yield the OPRF output.
        """
        
        # 1. Air Traffic Control: Run the ZKP verification protocol
        is_verified = Authentication_Bag.Check_Input(self.Face_Authenticator)
        
        if not is_verified:
            return False

        # =====================================================================
        # OBLIVIOUS TRANSFER & OPRF PHASE
        # =====================================================================
        Authentication_server=Authentication_Bag.bag_authenticator
        
        # 2. Server: Generate the OT boxes and offset using the verified bits
        R_list, S0_list, S1_list, G_out = Authentication_server.GenerateOPRFData(python_bulletproofs.get_dalek_default_h())
        
        # 3. Client: Evaluate the OT using their hidden blinders to extract the point
        oprf_point_bytes = self.Face_Authenticator.EvaluateOPRF(R_list, S0_list, S1_list, G_out)
        
        # Returning the raw OPRF point for the next steps!
        self.sessonOPRF=oprf_point_bytes

    def derive_variants(self, registered_face) -> list:
        if self.LSH_hash is None:
            raise RuntimeError("[Client] Cannot derive variants: LSH is not initialized.")

        # 1. Calculate the Client's current base_hash
        raw_base_hash = self.LSH_hash.hash(registered_face, k_bits=12)
        
        # FIX: Invert the hash to match the ZKP "Sign-Bit" logic
        full_mask = (1 << self.LSH_hash.num_bits) - 1
        base_hash = raw_base_hash ^ full_mask

        # 2. EXPLICIT FIX: Run the filter to populate the indices!
        # self.LSH_hash.filter_by_hyperplane(registered_face, self.LSH_hash.tolerance, k_bits=12)
        # I commented out the previous one since the indices should already be initialized
        indices = self.LSH_hash.filtered_indices 
        
        # 3. Reverse the index because CosineLSH hashes MSB-first
        shifts = [int((self.LSH_hash.num_bits - 1) - idx) for idx in indices]
        
        # 4. Prepare the base template: Clear the bits at the unreliable positions
        base_template = base_hash
        for shift in shifts:
            base_template &= ~(1 << shift)  
            
        variants = []
        num_combinations = 1 << len(shifts) 

        # 5. Generate all combinations using a binary counter
        for i in range(num_combinations):
            current_variant = base_template
            for j, shift in enumerate(shifts):
                # If the j-th bit of our counter 'i' is 1, set the bit in the variant
                if (i >> j) & 1:
                    current_variant |= (1 << shift)
                    
            variants.append(current_variant)

        return variants
    
    def get_Oprf_Keys(self, recoveredMasterKey: bytes, master_cipher: bytes):
        """
        Decrypts the Master Ciphertext and parses out the Server's persistent 
        Zero-Knowledge OPRF keys (k_0 and k_1 ... k_M).
        """
        # (Assuming AuthenticatedEncryption is available in the Client's scope)
        AE = AuthenticatedEncryption()
        
        try:
            # 1. Decrypt the heavy envelope payload
            server_keys_payload = AE.Decrypt(recoveredMasterKey, master_cipher)
        except Exception as e:
            raise RuntimeError("[Client] ❌ Failed to decrypt Master Ciphertext! Invalid Master Key.")

        # 2. Extract k_0 (The first 32 bytes)
        k_0 = int.from_bytes(server_keys_payload[0:32], byteorder='little')
        
        # 3. Extract the remaining M keys (32 bytes each)
        M = self.LSH_hash.num_bits
        k_keys = []
        
        offset = 32
        for i in range(M):
            chunk = server_keys_payload[offset : offset + 32]
            k_i = int.from_bytes(chunk, byteorder='little')
            k_keys.append(k_i)
            offset += 32
        
        return k_0, k_keys
    
    def RawOPRF(self,k_0,k_keys, pw_variant:int):
        
        """
        Locally computes the PRF output for a given integer variant.
        Mirrors the Server's direct evaluation to independently audit the DB.
        """
        # The Curve25519 Prime Order
        L = (1 << 252) + 27742317777372353535851937790883648493
        M = self.LSH_hash.num_bits
        
        # Start the accumulator at the base key k_0
        accumulator = k_0
        
        for i in range(M):
            # Read the bits exactly how the server read them!
            # (If your server used the standard LSB-first mapping):
            shift_amount = (M - 1) - i
            bit_is_set = (pw_variant >> shift_amount) & 1   
            
            if bit_is_set == 1:
                # ADDITION ROUTE: Add the corresponding key modulo L
                accumulator = (accumulator + k_keys[i]) % L

        # Map the final accumulated scalar back to a geometric point on the curve
        # (Assuming int_to_scalar_bytes and scalar_to_point are in your global scope)
        prf_scalar_bytes = int_to_scalar_bytes(accumulator)
        final_prf_point = scalar_to_point(prf_scalar_bytes)
        
        return final_prf_point
    
    def audit_all_bags(self, pake_server):
        """
        Post-authentication method. Once the client recovers their pristine face 
        from ONE successful bag, they use it to interactively audit ALL bags on the server.
        """
        if self.original_face_vector is None:
            print("[Client] ❌ Cannot audit server: Must successfully authenticate first to recover the pristine face.")
            return False
        
        # 1. Temporarily swap the noisy face for the pristine center face. 
        # This guarantees 100% success on the OT phase for every bag!
        original_noisy_face = self.face
        self.face = CosineLSH.quantize_array(self.original_face_vector, 12)
        
        for idx, server_bag in enumerate(pake_server.bags):
            print(f"\n[Client] --- Auditing Disjoint Bag {idx + 1}/{len(pake_server.bags)} ---")
            
            # 2. Re-initialize ZKP backend for THIS specific bag
            self.Initialize(server_bag.LSH_hash)
            
            # 3. Run the interactive ZKP and OT to unlock the bag's specific master cipher
            self.getOPRF(server_bag)
            
            if not self.sessonOPRF:
                raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: ZKP rejected for Bag {idx+1}. Server is cheating!")
                
            # 4. Derive Tag and Key
            tag_bytes = hashlib.sha256(b"TAG" + self.sessonOPRF).digest()
            tag_str = tag_bytes.hex()
            ek_i = hashlib.sha256(b"KEY" + self.sessonOPRF).digest()
            
            # 5. Check if the pristine center exists in the DB
            if tag_str not in server_bag.DB:
                raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: Pristine face tag missing in Bag {idx+1}!")
                
            # 6. Decrypt the payload to get THIS bag's master key
            AE = AuthenticatedEncryption()
            try:
                decrypted_payload = AE.Decrypt(ek_i, server_bag.DB[tag_str])
            except Exception:
                raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: Invalid ciphertext in Bag {idx+1}!")
                
            recovered_face_bytes = decrypted_payload[:-32]
            recovered_master_key = decrypted_payload[-32:]
            
            # 7. Decrypt the Master Cipher to get THIS bag's OPRF keys
            OPRF_k0, OPRF_keys = self.get_Oprf_Keys(recovered_master_key, server_bag.master_cipher)
            
            # 8. Locally construct all variants and audit the DB
            variants = self.derive_variants(self.original_face_vector)
            
            for variant in variants:
                local_prf_point = self.RawOPRF(OPRF_k0, OPRF_keys, variant)
                local_tag = hashlib.sha256(b"TAG" + local_prf_point).digest().hex()
                local_ek = hashlib.sha256(b"KEY" + local_prf_point).digest()
                
                if local_tag not in server_bag.DB:
                    raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: Server omitted variant {variant} in Bag {idx+1}!")
                    
                expected_payload = recovered_face_bytes + recovered_master_key
                try:
                    decrypted_audit_payload = AE.Decrypt(local_ek, server_bag.DB[local_tag])
                    if decrypted_audit_payload != expected_payload:
                        raise ValueError()
                except Exception:
                    raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: Tampered payload for variant in Bag {idx+1}!")

        # Restore the client's original noisy face state so no data is permanently altered
        self.face = original_noisy_face
        return True
    
    def verify(self,server_bag:PAKE_Bag, main_server:PAKE_server):
        # we first initialize and do the verification 
        self.Initialize(server_bag.LSH_hash)
        self.getOPRF(server_bag)

        # If ZKP or OT failed, self.sessonOPRF will be None
        if not self.sessonOPRF:
            print("[Client] ❌ Authentication aborted. Zero-Knowledge Proofs failed.")
            return False

        # 2. Key Derivation Function (KDF) using Domain Separation
        # Derive the dictionary Tag using the exact same prefix as the Server
        tag_bytes = hashlib.sha256(b"TAG" + self.sessonOPRF).digest()
        tag_str = tag_bytes.hex()
        
        # Derive the 32-byte symmetric encryption key
        ek_i = hashlib.sha256(b"KEY" + self.sessonOPRF).digest()

        # =====================================================================
        # DATABASE LOOKUP & DECRYPTION
        # =====================================================================
        
        # 3. Check the DB for our Tag
        database = server_bag.DB

        if tag_str not in database:
            print("[Client] ❌ Authentication Failed: Tag not found in DB. Face vector is outside the error ball.")
            return False
            
        ciphertext = database[tag_str]

        print("Face Exist, Authenticated [DEBUG MESSAGE]")

        # 4. Decrypt the payload
        # (Assuming your Client has access to the AuthenticatedEncryption class)
        AE = AuthenticatedEncryption()
        
        try:
            # If the ek_i is wrong, or the ciphertext was tampered with, this will throw an error
            decrypted_payload = AE.Decrypt(ek_i, ciphertext)
        except Exception as e:
            print("[Client] ❌ Decryption Failed: Invalid key or corrupted ciphertext.")
            return False

        # 5. Parse the Payload (register_face_hash_bytes || master_key)
        # hash_byte_len = (self.LSH_hash.num_bits + 7) // 8
        recovered_face_bytes = decrypted_payload[:-32]
        recovered_master_key = decrypted_payload[-32:]

        # to do the rest bag audit, make sure we save the current face vector
        self.original_face_vector = np.frombuffer(recovered_face_bytes, dtype=np.float32)

        return self.audit_all_bags(main_server)

        # variants=self.derive_variants(self.original_face_vector)
        # OPRF_k0, OPRF_keys=self.get_Oprf_Keys(recovered_master_key,server_bag.master_cipher)

        # # now perform the final check by reconsturcting the DB and check

        # for variant in variants:
        #     # A. Calculate the PRF locally
        #     local_prf_point = self.RawOPRF(OPRF_k0, OPRF_keys, variant)
            
        #     # B. Derive Tag and Encryption Key
        #     local_tag = hashlib.sha256(b"TAG" + local_prf_point).digest().hex()
        #     local_ek = hashlib.sha256(b"KEY" + local_prf_point).digest()
            
        #     # C. Check if the Server actually created this entry
        #     if local_tag not in server_bag.DB:
        #         raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: Server omitted variant {variant}!")
                
        #     # D. Ensure the cipher decrypts correctly to the expected payload
        #     expected_payload = recovered_face_bytes + recovered_master_key
        #     try:
        #         decrypted_audit_payload = AE.Decrypt(local_ek, server_bag.DB[local_tag])
        #         if decrypted_audit_payload != expected_payload:
        #             raise ValueError("Payload mismatch")
        #     except Exception:
        #         raise RuntimeError(f"[Client] 🚨 AUDIT FAILED: Server tampered with the ciphertext for tag {local_tag[:8]}!")

        # print("[Client] 🛡️ Audit Passed! The Server is honest, and the Error Ball is perfectly constructed.")


# 2. Experiment and Sanity Checks #

## 2.0 Helper Functions and Facial Dataset Load ##

Load Facial Dataset

In [14]:
# Just in case we have additional dataset to test on
folder_extracted="train"

# define the official data structure in RAM 

facial_data=dict()

def restore(obj):
    if isinstance(obj, list):
        # If it's a list of numbers, convert to numpy array
        if all(isinstance(x, (int, float)) for x in obj):
            return np.array(obj)
        # Otherwise recurse element-wise
        return [restore(x) for x in obj]
    if isinstance(obj, dict):
        return {k: restore(v) for k, v in obj.items()}
    return obj

with open(f"data_{folder_extracted}.json", "r") as f:
    facial_data_loaded = json.load(f)

facial_data= restore(facial_data_loaded)

Raw Quantization to certain bits (For 2.2 section)

In [15]:
def quantize_to_12bit(float_vector: np.ndarray) -> np.ndarray:
    """
    Safely scales a float vector [-1.0, 1.0] to a 12-bit signed integer vector [-2047, 2047].
    """
    # Clip to ensure no outliers break the math
    clipped = np.clip(float_vector, -1.0, 1.0)
    # Scale by 2047 and round to nearest integer
    quantized = np.round(clipped * 2047).astype(int)
    return quantized

VECTOR_DIM = 128
NUM_HYPERPLANES = 64

## 2.1 Whole routine Testing (Without OPRF) ##

Timing test For CLIENT-SERVER functions

In [14]:
def run_zklsh_benchmark(facial_data_dict: dict, num_iterations: int = 10, output_file: str = "BulletProof_Routine_benchmarks.csv"):
    print(f"\n🚀 Starting Zero-Knowledge LSH Benchmark ({num_iterations} iterations)...")
    
    VECTOR_DIM = 128
    NUM_HYPERPLANES = 64
    
    # We will track timings in a dictionary of lists
    timings = {
        "Client_1_CommitFace": [],
        "Server_1_VerifyFace": [],
        "Client_2_DotCommitment": [],
        "Server_2_DotCommitment": [],
        "Client_3_BitCommitment": [],
        "Server_3_ReceiveBits": [],
        "Client_4_LinkageProof": [],
        "Server_4_VerifyLinkage": [],
        "Total_Client_Time": [],
        "Total_Server_Time": [],
        "Total_Protocol_Time": []
    }

    person_keys = list(facial_data_dict.keys())

    for i in range(num_iterations):
        print(f"   -> Running iteration {i+1}/{num_iterations}...")
        
        # --- 1. Fresh Data Setup ---
        # Grab a random person and a random face from their array
        random_person = random.choice(person_keys)
        raw_face = random.choice(facial_data_dict[random_person])
        raw_hyper = np.random.uniform(-1.0, 1.0, (NUM_HYPERPLANES, VECTOR_DIM))
        
        quantized_face = quantize_to_12bit(raw_face)
        quantized_hyper = quantize_to_12bit(raw_hyper)
        
        client = FaceAuthenticationClient(quantized_face, quantized_hyper)
        server = FaceAuthenticationServer(quantized_hyper)
        
        client_time_total = 0.0
        server_time_total = 0.0

        # --- STEP 1 ---
        t0 = time.perf_counter()
        face_comms, face_proofs = client.CommitFace()
        t_client1 = time.perf_counter() - t0
        client_time_total += t_client1
        timings["Client_1_CommitFace"].append(t_client1)

        t0 = time.perf_counter()
        is_face_valid = server.VerifyFaceCommitments(face_comms, face_proofs)
        t_server1 = time.perf_counter() - t0
        server_time_total += t_server1
        timings["Server_1_VerifyFace"].append(t_server1)
        
        if not is_face_valid: raise RuntimeError("Protocol Failed at Step 1.")

        # --- STEP 2 ---
        t0 = time.perf_counter()
        client.CreateDotCommitment()
        t_client2 = time.perf_counter() - t0
        client_time_total += t_client2
        timings["Client_2_DotCommitment"].append(t_client2)

        t0 = time.perf_counter()
        server.CreateDotCommitment()
        t_server2 = time.perf_counter() - t0
        server_time_total += t_server2
        timings["Server_2_DotCommitment"].append(t_server2)

        # --- STEP 3 ---
        t0 = time.perf_counter()
        bit_comms = client.CreateBitCommitment(DALEK_H)
        t_client3 = time.perf_counter() - t0
        client_time_total += t_client3
        timings["Client_3_BitCommitment"].append(t_client3)

        t0 = time.perf_counter()
        server.ReceiveBitCommitments(bit_comms)
        t_server3 = time.perf_counter() - t0
        server_time_total += t_server3
        timings["Server_3_ReceiveBits"].append(t_server3)

        # --- STEP 4 ---
        t0 = time.perf_counter()
        linkage_comms, linkage_proofs = client.CreateLinkageProof()
        t_client4 = time.perf_counter() - t0
        client_time_total += t_client4
        timings["Client_4_LinkageProof"].append(t_client4)

        t0 = time.perf_counter()
        is_linkage_valid = server.VerifyLinkageProofs(linkage_proofs)
        t_server4 = time.perf_counter() - t0
        server_time_total += t_server4
        timings["Server_4_VerifyLinkage"].append(t_server4)
        
        if not is_linkage_valid: raise RuntimeError("Protocol Failed at Step 4.")

        # --- Totals ---
        timings["Total_Client_Time"].append(client_time_total)
        timings["Total_Server_Time"].append(server_time_total)
        timings["Total_Protocol_Time"].append(client_time_total + server_time_total)

    # --- 4. Process and Export to CSV ---
    print(f"\n📊 Benchmarking complete! Exporting results to {output_file}...")
    
    with open(output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        # Write CSV Headers
        writer.writerow(["Operation", "Mean_Time_(s)", "Std_Dev_(s)", "Min_Time_(s)", "Max_Time_(s)"])
        
        for operation, times in timings.items():
            mean_time = statistics.mean(times)
            std_dev = statistics.stdev(times) if len(times) > 1 else 0.0
            min_time = min(times)
            max_time = max(times)
            
            writer.writerow([
                operation, 
                f"{mean_time:.6f}", 
                f"{std_dev:.6f}", 
                f"{min_time:.6f}", 
                f"{max_time:.6f}"
            ])
            
            # Print to console for immediate satisfaction
            print(f"{operation.ljust(30)} | Mean: {mean_time:.4f}s")

    print(f"\n✅ All done. Open '{output_file}' to view the detailed metrics.")

In [ ]:
run_zklsh_benchmark(facial_data, num_iterations=10)

## 2.2 Whole routine Testing (With OPRF) ##

In [10]:
def run_full_protocol_benchmark(facial_data_dict: dict, num_iterations: int = 10, output_file: str = "full_protocol_benchmarks.csv"):

    VECTOR_DIM = 128
    NUM_HYPERPLANES = 64
    
    # Track timings for all 5 stages
    timings = {
        "Client_1_CommitFace": [],
        "Server_1_VerifyFace": [],
        "Client_2_DotCommitment": [],
        "Server_2_DotCommitment": [],
        "Client_3_BitCommitment": [],
        "Server_3_ReceiveBits": [],
        "Client_4_LinkageProof": [],
        "Server_4_VerifyLinkage": [],
        "Server_5_GenerateOPRF": [],  # NEW: OT Setup
        "Client_5_EvaluateOPRF": [],  # NEW: OT Eval
        "Total_Client_Time": [],
        "Total_Server_Time": [],
        "Total_Protocol_Time": []
    }

    person_keys = list(facial_data_dict.keys())

    for i in range(num_iterations):
        print(f"\n   ---> Running iteration {i+1}/{num_iterations}...")
        
        # --- 1. Fresh Data Setup ---
        random_person = random.choice(person_keys)
        person_faces = facial_data_dict[random_person]
        random_face_idx = random.randint(0, len(person_faces) - 1)
        
        raw_face = np.array(person_faces[random_face_idx]).flatten()

        raw_hyper = np.random.uniform(-1.0, 1.0, (NUM_HYPERPLANES, VECTOR_DIM))
        quantized_face = quantize_to_12bit(raw_face)
        quantized_hyper = quantize_to_12bit(raw_hyper)
        
        client = FaceAuthenticationClient(quantized_face, quantized_hyper)
        server = FaceAuthenticationServer(quantized_hyper)
        
        client_time_total = 0.0
        server_time_total = 0.0

        # --- STEP 1: Face Commitments ---
        t0 = time.perf_counter()
        face_comms, face_proofs = client.CommitFace()
        client_time_total += (time.perf_counter() - t0); timings["Client_1_CommitFace"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        is_face_valid = server.VerifyFaceCommitments(face_comms, face_proofs)
        server_time_total += (time.perf_counter() - t0); timings["Server_1_VerifyFace"].append(time.perf_counter() - t0)
        if not is_face_valid: raise RuntimeError("Failed at Step 1.")

        # --- STEP 2: Dot Products ---
        t0 = time.perf_counter()
        client.CreateDotCommitment()
        client_time_total += (time.perf_counter() - t0); timings["Client_2_DotCommitment"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        server.CreateDotCommitment()
        server_time_total += (time.perf_counter() - t0); timings["Server_2_DotCommitment"].append(time.perf_counter() - t0)

        # --- STEP 3: Bit Commitments ---
        t0 = time.perf_counter()
        bit_comms = client.CreateBitCommitment(DALEK_H)
        client_time_total += (time.perf_counter() - t0); timings["Client_3_BitCommitment"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        server.ReceiveBitCommitments(bit_comms)
        server_time_total += (time.perf_counter() - t0); timings["Server_3_ReceiveBits"].append(time.perf_counter() - t0)

        # --- STEP 4: Linkage Proofs ---
        t0 = time.perf_counter()
        linkage_comms, linkage_proofs = client.CreateLinkageProof()
        client_time_total += (time.perf_counter() - t0); timings["Client_4_LinkageProof"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        is_linkage_valid = server.VerifyLinkageProofs(linkage_proofs)
        server_time_total += (time.perf_counter() - t0); timings["Server_4_VerifyLinkage"].append(time.perf_counter() - t0)
        if not is_linkage_valid: raise RuntimeError("Failed at Step 4.")

        # --- STEP 5: OT & OPRF Phase (NEW) ---
        t0 = time.perf_counter()
        R_list, S0_list, S1_list, G_out = server.GenerateOPRFData(DALEK_H)
        server_time_total += (time.perf_counter() - t0); timings["Server_5_GenerateOPRF"].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        final_oprf_point = client.EvaluateOPRF(R_list, S0_list, S1_list, G_out)
        client_time_total += (time.perf_counter() - t0); timings["Client_5_EvaluateOPRF"].append(time.perf_counter() - t0)
        
        # Print the final resulting point (just the first 16 hex chars to keep the console clean)
        print(f"      💎 Final OPRF Output: {final_oprf_point.hex()[:16]}...{final_oprf_point.hex()[-16:]}")

        # --- Totals ---
        timings["Total_Client_Time"].append(client_time_total)
        timings["Total_Server_Time"].append(server_time_total)
        timings["Total_Protocol_Time"].append(client_time_total + server_time_total)

    # --- Process and Export to CSV ---
    print(f"\n📊 Benchmarking complete! Exporting results to {output_file}...")
    
    with open(output_file, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Operation", "Mean_Time_(s)", "Mean_Time_(ms)", "Std_Dev_(s)", "Min_Time_(s)", "Max_Time_(s)"])
        
        for operation, times in timings.items():
            mean_time = statistics.mean(times)
            mean_ms = mean_time * 1000  # Convert to milliseconds for easier reading
            std_dev = statistics.stdev(times) if len(times) > 1 else 0.0
            min_time = min(times)
            max_time = max(times)
            
            writer.writerow([
                operation, 
                f"{mean_time:.6f}", 
                f"{mean_ms:.3f}", 
                f"{std_dev:.6f}", 
                f"{min_time:.6f}", 
                f"{max_time:.6f}"
            ])
            
            print(f"{operation.ljust(30)} | Mean: {mean_time:.4f}s ({mean_ms:.1f} ms)")

    print(f"\n✅ All done. Detailed metrics saved to '{output_file}'.")

In [11]:
run_full_protocol_benchmark(facial_data, num_iterations=30)


   ---> Running iteration 1/30...

[Server] Verifying 128 Bulletproofs (16-bit capacity)...
[Server] ✅ ACCEPTED: All face vector commitments cryptographically verified!

[Server] ✅ Received and saved 64 Bit Commitments from the client.

[Server] Computing Linkage Commitments and verifying 64 32-bit ZKPs...
[Server] ✅ ACCEPTED: All Linkage Proofs verified! The LSH bits mathematically match the true dot products.
[Server] ✅ OT Setup Complete. Sending 64 tuples to Client.
      💎 Final OPRF Output: a8af572efa70dc2e...dc96daaf73fad927

   ---> Running iteration 2/30...

[Server] Verifying 128 Bulletproofs (16-bit capacity)...
[Server] ✅ ACCEPTED: All face vector commitments cryptographically verified!

[Server] ✅ Received and saved 64 Bit Commitments from the client.

[Server] Computing Linkage Commitments and verifying 64 32-bit ZKPs...
[Server] ✅ ACCEPTED: All Linkage Proofs verified! The LSH bits mathematically match the true dot products.
[Server] ✅ OT Setup Complete. Sending 64 tuple

## 2.3 FACIAL PAKE Testing ##

### 2.3.1 Single Bag Unit Test ###

1. Get a random face tested for registration

In [16]:
person_keys = list(facial_data.keys())
random_person = random.choice(person_keys)
raw_face = random.choice(facial_data[random_person])

In [32]:
print("==================================================")
print("🚀 Starting PAKE Bag Registration Test")
print("==================================================")

# 2. Setup Parameters
# We use a tolerance of 8 (2^8 = 256 variants) for a fast, readable test.
# You can bump this to 10 or 12 for production stress-testing!
M_BITS = 64
TOLERANCE = 12
DISJOINT_BAG= 3

print(f"Initializing PAKE Bag (Hyperplanes: {M_BITS}, Tolerance: {TOLERANCE})...")
server=PAKE_server(DISJOINT_BAG,number_of_hyperplanes=M_BITS,tolerance=TOLERANCE)

# 3. Execute and Time the Registration
start_time = time.time()

success = server.register(raw_face)

end_time = time.time()

# 4. Verification and Metrics
print("\n==================================================")
print("📊 Registration Verification Metrics")
print("==================================================")

if success:
    print("Status: ✅ Registration function returned True")
else:
    print("Status: ❌ Registration failed")

print(f"\n⏱️ Total Registration Time: {end_time - start_time:.4f} seconds")
print("==================================================")

🚀 Starting PAKE Bag Registration Test
Initializing PAKE Bag (Hyperplanes: 64, Tolerance: 12)...

📊 Registration Verification Metrics
Status: ✅ Registration function returned True

⏱️ Total Registration Time: 1.0342 seconds


In [34]:
print("\n==================================================")
print("🔐 Starting PAKE Client Authentication Test")
print("==================================================")
test_face= random.choice(facial_data[random_person])
# 1. Initialize the Client with the EXACT same face used for registration
print("[Test] Initializing PAKE Client with the registered face...")
client = PAKE_client(test_face)

# 2. Execute and Time the Authentication
start_auth_time = time.time()

# The verify function handles Initialize(LSH), getOPRF(server), 
# Tag Derivation, DB Lookup, and Master Cipher decryption!
auth_success = server.authenticate(client)

end_auth_time = time.time()

# 3. Authentication Metrics
print("\n==================================================")
print("📊 Authentication Verification Results")
print("==================================================")

if auth_success:
    print("Status: ✅ Authentication function returned True")
    print("Result: 🎉 Successfully authenticated and unlocked the Master Cipher!")
else:
    print("Status: ❌ Authentication failed")

print(f"\n⏱️ Total Authentication Time: {end_auth_time - start_auth_time:.4f} seconds")
print("==================================================")


🔐 Starting PAKE Client Authentication Test
[Test] Initializing PAKE Client with the registered face...

[Protocol] 🔄 Starting Zero-Knowledge Verification...

[Server] Verifying 128 Bulletproofs (16-bit capacity)...
Face Exist, Authenticated [DEBUG MESSAGE]

[Client] --- Auditing Disjoint Bag 1/3 ---

[Protocol] 🔄 Starting Zero-Knowledge Verification...

[Server] Verifying 128 Bulletproofs (16-bit capacity)...


TypeError: bad operand type for abs(): 'NoneType'